<a href="https://colab.research.google.com/github/MR-just01/Llama3.2-Reasoning/blob/main/notebooks/standardized_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Cleaning and Standardization

## Objective

This notebook transforms the raw reasoning datasets into a unified instruction-following format suitable for fine-tuning Llama 3.2 using QLoRA.

Datasets covered:

- GSM8K
- StrategyQA
- ARC Challenge
- AQUA-RAT

For each dataset, the following steps are performed:

1. Load the dataset
2. Remove unnecessary fields
3. Extract reasoning (if available)
4. Extract the final answer
5. Add instruction prompts
6. Standardize column names
7. Save the cleaned dataset

# GSM8K Standardization

In [1]:
import pandas as pd
import re

from datasets import load_dataset

In [2]:
gsm8k = load_dataset("openai/gsm8k", "main")

gsm_df = gsm8k["train"].to_pandas()

print(gsm_df.shape)

gsm_df.head()

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

(7473, 2)


,question,answer
0,Natalia sold clips to 48 of her friends in Apr...,Natalia sold 48/2 = <<48/2=24>>24 clips in May...
1,Weng earns $12 an hour for babysitting. Yester...,Weng earns 12/60 = $<<12/60=0.2>>0.2 per minut...
2,Betty is saving money for a new wallet which c...,"In the beginning, Betty has only 100 / 2 = $<<..."
3,"Julie is reading a 120-page book. Yesterday, s...",Maila read 12 x 2 = <<12*2=24>>24 pages today....
4,James writes a 3-page letter to 2 different fr...,He writes each friend 3*2=<<3*2=6>>6 pages a w...


In [3]:
gsm_clean = gsm_df.copy()

In [4]:
gsm_clean["question"] = gsm_clean["question"].str.strip()

gsm_clean["answer"] = gsm_clean["answer"].str.strip()

In [5]:
def extract_final_answer(answer):

    match = re.search(r"####\s*(.*)", answer)

    if match:
        return match.group(1).strip()

    return None

# Save the original answer column
gsm_clean["full_solution"] = gsm_clean["answer"]

In [6]:
gsm_clean["reasoning"] = (
    gsm_clean["full_solution"]
    .str.replace(r"####.*", "", regex=True)
    .str.strip()
)

In [7]:
import re

def extract_final_answer(solution):
    match = re.search(r"####\s*(.*)", solution)
    if match:
        return match.group(1).strip()
    return None

gsm_clean["answer"] = gsm_clean["full_solution"].apply(extract_final_answer)

In [8]:
gsm_clean.drop(columns=["full_solution"], inplace=True)

In [9]:
gsm_clean["instruction"] = "Solve the following math reasoning problem step by step."

In [11]:
gsm_clean.rename(
    columns={
        "question": "input"
    },
    inplace=True
)

In [12]:
# added meta data
gsm_clean["dataset"] = "gsm8k"
gsm_clean["task_type"] = "math_reasoning"

In [13]:
gsm_clean = gsm_clean[
    [
        "instruction",
        "input",
        "reasoning",
        "answer",
        "dataset",
        "task_type"
    ]
]

In [14]:
gsm_clean.head()

,instruction,input,reasoning,answer,dataset,task_type
0,Solve the following math reasoning problem ste...,Natalia sold clips to 48 of her friends in Apr...,Natalia sold 48/2 = <<48/2=24>>24 clips in May...,72,gsm8k,math_reasoning
1,Solve the following math reasoning problem ste...,Weng earns $12 an hour for babysitting. Yester...,Weng earns 12/60 = $<<12/60=0.2>>0.2 per minut...,10,gsm8k,math_reasoning
2,Solve the following math reasoning problem ste...,Betty is saving money for a new wallet which c...,"In the beginning, Betty has only 100 / 2 = $<<...",5,gsm8k,math_reasoning
3,Solve the following math reasoning problem ste...,"Julie is reading a 120-page book. Yesterday, s...",Maila read 12 x 2 = <<12*2=24>>24 pages today....,42,gsm8k,math_reasoning
4,Solve the following math reasoning problem ste...,James writes a 3-page letter to 2 different fr...,He writes each friend 3*2=<<3*2=6>>6 pages a w...,624,gsm8k,math_reasoning


In [15]:
gsm_clean.isnull().sum()

,0
instruction,0
input,0
reasoning,0
answer,0
dataset,0
task_type,0


## GSM8K Standardization Summary

The GSM8K dataset was transformed into the common instruction-following format.

Performed preprocessing:

- Trimmed whitespace
- Extracted reasoning steps
- Extracted final numeric answer
- Added instruction prompt
- Added dataset metadata
- Renamed columns to match the unified schema
- Saved standardized dataset

In [16]:
gsm_clean.to_csv(
    "gsm8k_standardized.csv",
    index=False
)

print("✅ GSM8K standardized dataset saved successfully.")

✅ GSM8K standardized dataset saved successfully.


In [21]:
print(gsm_clean.columns.tolist())
gsm_clean.head(1).T

['instruction', 'input', 'reasoning', 'answer', 'dataset', 'task_type']


,0
instruction,Solve the following math reasoning problem ste...
input,Natalia sold clips to 48 of her friends in Apr...
reasoning,Natalia sold 48/2 = <<48/2=24>>24 clips in May...
answer,72
dataset,gsm8k
task_type,math_reasoning


In [22]:

print(gsm_clean.columns)

Index(['instruction', 'input', 'reasoning', 'answer', 'dataset', 'task_type'], dtype='object')


In [23]:
check = pd.read_csv("gsm8k_standardized.csv")

print(check.head(1).T)

                                                             0
instruction  Solve the following math reasoning problem ste...
input        Natalia sold clips to 48 of her friends in Apr...
reasoning    Natalia sold 48/2 = <<48/2=24>>24 clips in May...
answer                                                      72
dataset                                                  gsm8k
task_type                                       math_reasoning


In [24]:
from google.colab import files

files.download("gsm8k_standardized.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
gsm_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7473 entries, 0 to 7472
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  7473 non-null   object
 1   input        7473 non-null   object
 2   reasoning    7473 non-null   object
 3   answer       7473 non-null   object
 4   dataset      7473 non-null   object
 5   task_type    7473 non-null   object
dtypes: object(6)
memory usage: 350.4+ KB


# **StrategyQA**

In [ ]:
import pandas as pd
import re



In [ ]:
from datasets import load_dataset

strategyqa = load_dataset("ChilleD/StrategyQA")

strategy_df = strategyqa["train"].to_pandas()

strategy_df.head()

data/train-00000-of-00001-506370352f6228(…): reconstructing file:   0%|          |  0.00B /  369kB            

data/train-00000-of-00001-506370352f6228(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-bae602f3ee37f4c(…): reconstructing file:   0%|          |  0.00B /  161kB            

data/test-00000-of-00001-bae602f3ee37f4c(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1603 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/687 [00:00<?, ? examples/s]

,qid,term,description,question,answer,facts
0,4fd64bb6ce5b78ab20b6,Mixed martial arts,full contact combat sport,Is Mixed martial arts totally original from Ro...,False,Mixed Martial arts in the UFC takes place in a...
1,f378f856bdaff39cdfa3,Cuisine of Hawaii,Cuisine of Hawaii,Is the cuisine of Hawaii suitable for a vegan?,False,"Per capita, Hawaiians are the second largest ..."
2,4e1b65e81ec09397b26e,Giant squid,Deep-ocean dwelling squid in the family Archit...,Is capturing giant squid in natural habitat im...,True,"Giant squids live between 1,000 and 3,800 feet..."
3,6d14da7484991bf588cf,Royal Air Force,Aerial warfare service branch of the British A...,Did the Royal Air Force fight in the Boxer Reb...,False,The Boxer Rebellion took place from 1899–1901 ...
4,3d01af5db202bc7d33b9,Eggplant,plant species Solanum melongena,Would someone in Mumbai refer to Solanum melon...,False,Mumbia is a city in India. India is a country ...


In [ ]:
strategy_clean = strategy_df.copy()

In [ ]:
strategy_clean = strategy_clean[
    [
        "question",
        "facts",
        "answer"
    ]
]

strategy_clean.head()

,question,facts,answer
0,Is Mixed martial arts totally original from Ro...,Mixed Martial arts in the UFC takes place in a...,False
1,Is the cuisine of Hawaii suitable for a vegan?,"Per capita, Hawaiians are the second largest ...",False
2,Is capturing giant squid in natural habitat im...,"Giant squids live between 1,000 and 3,800 feet...",True
3,Did the Royal Air Force fight in the Boxer Reb...,The Boxer Rebellion took place from 1899–1901 ...,False
4,Would someone in Mumbai refer to Solanum melon...,Mumbia is a city in India. India is a country ...,False


In [ ]:
strategy_clean.info()
strategy_clean.isnull().sum()
strategy_clean["question"].duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1603 entries, 0 to 1602
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  1603 non-null   object
 1   facts     1603 non-null   object
 2   answer    1603 non-null   bool  
dtypes: bool(1), object(2)
memory usage: 26.7+ KB


,0
question,0
facts,0
answer,0


In [ ]:
strategy_clean["answer"] = strategy_clean["answer"].map({
    True: "Yes",
    False: "No"
})
strategy_clean["answer"].value_counts()

,count
answer,
No,865
Yes,738


Rename the dataset to match  unified schema

In [ ]:
strategy_clean = strategy_clean.rename(
    columns={
        "question": "input",
        "facts": "reasoning",
        "answer": "answer"
    }
)

In [ ]:
strategy_clean["instruction"] = (
    "Read the question and supporting facts carefully. "
    "Reason about the information and answer with Yes or No."
)

## Add metadata
strategy_clean["dataset"] = "StrategyQA"

strategy_clean["task_type"] = "commonsense_reasoning"

In [ ]:
strategy_clean = strategy_clean[
    [
        "instruction",
        "input",
        "reasoning",
        "answer",
        "dataset",
        "task_type"
    ]
]

In [ ]:
strategy_clean.info()
strategy_clean.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1603 entries, 0 to 1602
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  1603 non-null   object
 1   input        1603 non-null   object
 2   reasoning    1603 non-null   object
 3   answer       1603 non-null   object
 4   dataset      1603 non-null   object
 5   task_type    1603 non-null   object
dtypes: object(6)
memory usage: 75.3+ KB


,instruction,input,reasoning,answer,dataset,task_type
0,Read the question and supporting facts careful...,Is Mixed martial arts totally original from Ro...,Mixed Martial arts in the UFC takes place in a...,No,StrategyQA,commonsense_reasoning
1,Read the question and supporting facts careful...,Is the cuisine of Hawaii suitable for a vegan?,"Per capita, Hawaiians are the second largest ...",No,StrategyQA,commonsense_reasoning
2,Read the question and supporting facts careful...,Is capturing giant squid in natural habitat im...,"Giant squids live between 1,000 and 3,800 feet...",Yes,StrategyQA,commonsense_reasoning
3,Read the question and supporting facts careful...,Did the Royal Air Force fight in the Boxer Reb...,The Boxer Rebellion took place from 1899–1901 ...,No,StrategyQA,commonsense_reasoning
4,Read the question and supporting facts careful...,Would someone in Mumbai refer to Solanum melon...,Mumbia is a city in India. India is a country ...,No,StrategyQA,commonsense_reasoning


In [ ]:
strategy_clean.to_csv(
    "startegyQA_standardized.csv",
    index=False
)

print("✅ StartegyQA standardized dataset saved successfully.")

✅ StartegyQA standardized dataset saved successfully.


In [ ]:
from google.colab import files

files.download("startegyQA_standardized.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **ARC dataset**

In [ ]:
from datasets import load_dataset
import pandas as pd

arc = load_dataset("allenai/ai2_arc", "ARC-Challenge")

arc_df = arc["train"].to_pandas()

arc_df.head()

README.md:   0%|          | 0.00/9.00k [00:00<?, ?B/s]

ARC-Challenge/train-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B /  190kB            

ARC-Challenge/train-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

ARC-Challenge/test-00000-of-00001.parque(…): reconstructing file:   0%|          |  0.00B /  204kB            

ARC-Challenge/test-00000-of-00001.parque(…): downloading bytes:           |  0.00B            

...

ARC-Challenge/validation-00000-of-00001.(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1119 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1172 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

,id,question,choices,answerKey
0,Mercury_SC_415702,George wants to warm his hands quickly by rubb...,"{'text': ['dry palms', 'wet palms', 'palms cov...",A
1,MCAS_2009_5_6516,Which of the following statements best explain...,"{'text': ['The refrigerator door is smooth.', ...",B
2,Mercury_7233695,A fold observed in layers of sedimentary rock ...,"{'text': ['cooling of flowing magma.', 'conver...",B
3,Mercury_7041615,Which of these do scientists offer as the most...,"{'text': ['worldwide disease', 'global mountai...",D
4,Mercury_7041860,A boat is acted on by a river current flowing ...,"{'text': ['west', 'east', 'north', 'south'], '...",B


In [ ]:
arc_clean = arc_df.copy()

In [ ]:
arc_clean.loc[0, "choices"]

{'text': array(['dry palms', 'wet palms', 'palms covered with oil',
        'palms covered with lotion'], dtype=object),
 'label': array(['A', 'B', 'C', 'D'], dtype=object)}

In [ ]:
def format_choices(choice_dict):

    labels = choice_dict["label"]
    texts = choice_dict["text"]

    formatted = []

    for label, text in zip(labels, texts):
        formatted.append(f"{label}. {text}")

    return "\n".join(formatted)

In [ ]:
arc_clean["formatted_choices"] = arc_clean["choices"].apply(format_choices)
arc_clean[
    ["question", "formatted_choices"]
].head()

,question,formatted_choices
0,George wants to warm his hands quickly by rubb...,A. dry palms\nB. wet palms\nC. palms covered w...
1,Which of the following statements best explain...,A. The refrigerator door is smooth.\nB. The re...
2,A fold observed in layers of sedimentary rock ...,A. cooling of flowing magma.\nB. converging of...
3,Which of these do scientists offer as the most...,A. worldwide disease\nB. global mountain build...
4,A boat is acted on by a river current flowing ...,A. west\nB. east\nC. north\nD. south


In [ ]:
arc_clean["input"] = (
    arc_clean["question"]
    + "\n\nChoices:\n"
    + arc_clean["formatted_choices"]
)

In [ ]:
print(arc_clean.loc[0, "input"])

George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?

Choices:
A. dry palms
B. wet palms
C. palms covered with oil
D. palms covered with lotion


In [ ]:
print(arc_clean.loc[0, "formatted_choices"])

print(arc_clean.loc[0, "input"])

A. dry palms
B. wet palms
C. palms covered with oil
D. palms covered with lotion
George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?

Choices:
A. dry palms
B. wet palms
C. palms covered with oil
D. palms covered with lotion


In [ ]:
def extract_answer(row):

    labels = row["choices"]["label"]
    texts = row["choices"]["text"]

    mapping = dict(zip(labels, texts))

    return mapping[row["answerKey"]]

In [ ]:
arc_clean["answer"] = arc_clean.apply(extract_answer, axis=1)



arc_clean[["answerKey", "answer"]].head()


,answerKey,answer
0,A,dry palms
1,B,The refrigerator door contains iron.
2,B,converging of crustal plates.
3,D,impact of an asteroid created dust that blocke...
4,B,east


In [ ]:
arc_clean["instruction"] = (
    "Read the science question and the answer choices carefully. "
    "Reason about the options and provide the correct answer."
)

arc_clean["instruction"].head()

,instruction
0,Read the science question and the answer choic...
1,Read the science question and the answer choic...
2,Read the science question and the answer choic...
3,Read the science question and the answer choic...
4,Read the science question and the answer choic...


In [ ]:
arc_clean["reasoning"] = ""
arc_clean["reasoning"].head()

,reasoning
0,
1,
2,
3,
4,


In [ ]:
#add the metadata

arc_clean["dataset"] = "ARC-Challenge"
arc_clean["task_type"] = "science_reasoning"

arc_clean[["dataset", "task_type"]].head()

,dataset,task_type
0,ARC-Challenge,science_reasoning
1,ARC-Challenge,science_reasoning
2,ARC-Challenge,science_reasoning
3,ARC-Challenge,science_reasoning
4,ARC-Challenge,science_reasoning


In [ ]:
arc_clean = arc_clean[
    [
        "instruction",
        "input",
        "reasoning",
        "answer",
        "dataset",
        "task_type"
    ]
]

In [ ]:
arc_clean.info()
arc_clean.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1119 entries, 0 to 1118
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  1119 non-null   object
 1   input        1119 non-null   object
 2   reasoning    1119 non-null   object
 3   answer       1119 non-null   object
 4   dataset      1119 non-null   object
 5   task_type    1119 non-null   object
dtypes: object(6)
memory usage: 52.6+ KB


,instruction,input,reasoning,answer,dataset,task_type
0,Read the science question and the answer choic...,George wants to warm his hands quickly by rubb...,,dry palms,ARC-Challenge,science_reasoning
1,Read the science question and the answer choic...,Which of the following statements best explain...,,The refrigerator door contains iron.,ARC-Challenge,science_reasoning
2,Read the science question and the answer choic...,A fold observed in layers of sedimentary rock ...,,converging of crustal plates.,ARC-Challenge,science_reasoning
3,Read the science question and the answer choic...,Which of these do scientists offer as the most...,,impact of an asteroid created dust that blocke...,ARC-Challenge,science_reasoning
4,Read the science question and the answer choic...,A boat is acted on by a river current flowing ...,,east,ARC-Challenge,science_reasoning


In [ ]:
arc["reasoning"] = arc["reasoning"].fillna("")
arc_clean["reasoning"] = "Not provided"
arc["reasoning"].isnull().sum()
arc.dtypes

,0
instruction,object
input,object
reasoning,object
answer,object
dataset,object
task_type,object


In [ ]:
arc_clean.to_csv(
    "arc_challenge_standardized.csv",
    index=False
)

In [ ]:
from google.colab import files

files.download("arc_challenge_standardized.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

arc = pd.read_csv("arc_challenge_standardized.csv")

print("ARC")
print(arc.columns.tolist())
print(arc.dtypes)
print(arc.isnull().sum())

ARC
['instruction', 'input', 'reasoning', 'answer', 'dataset', 'task_type']
instruction    object
input          object
reasoning      object
answer         object
dataset        object
task_type      object
dtype: object
instruction    0
input          0
reasoning      0
answer         0
dataset        0
task_type      0
dtype: int64


In [ ]:
arc_clean["reasoning"].head()

,reasoning
0,Not provided
1,Not provided
2,Not provided
3,Not provided
4,Not provided


## **AQUA-RAT Standardization**

In [ ]:
from datasets import load_dataset

aqua = load_dataset("deepmind/aqua_rat")

aqua_df = aqua["train"].to_pandas()

aqua_df.head()

aqua_clean = aqua_df.copy()

README.md:   0%|          | 0.00/5.89k [00:00<?, ?B/s]

raw/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 25.4MB            

raw/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

raw/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 74.0kB            

raw/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

raw/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 76.1kB            

raw/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/97467 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/254 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/254 [00:00<?, ? examples/s]

In [ ]:
print(aqua_clean.shape)

aqua_clean.info()

aqua_clean.isnull().sum()

aqua_clean.head()

(97467, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97467 entries, 0 to 97466
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   question           97467 non-null  object
 1   options            97467 non-null  object
 2   rationale          97467 non-null  object
 3   correct            97467 non-null  object
 4   formatted_options  97467 non-null  object
 5   input              97467 non-null  object
 6   answer             97467 non-null  object
 7   reasoning          97467 non-null  object
 8   instruction        97467 non-null  object
dtypes: object(9)
memory usage: 6.7+ MB


,question,options,rationale,correct,formatted_options,input,answer,reasoning,instruction
0,"Two friends plan to walk along a 43-km trail, ...","[A)21, B)21.5, C)22, D)22.5, E)23]","If Q complete x kilometers, then P completes 1...",E,A. 21\nB. 21.5\nC. 22\nD. 22.5\nE. 23,"Two friends plan to walk along a 43-km trail, ...",23,"If Q complete x kilometers, then P completes 1...",Solve the following multiple-choice math reaso...
1,"In the coordinate plane, points (x, 1) and (5,...","[A)4 and 1, B)1 and 5, C)5 and 1, D)3 and 5, E...",Line k passes through the origin and has slope...,C,A. 4 and 1\nB. 1 and 5\nC. 5 and 1\nD. 3 and 5...,"In the coordinate plane, points (x, 1) and (5,...",5 and 1,Line k passes through the origin and has slope...,Solve the following multiple-choice math reaso...
2,"For all numbers p and q, the operation @ is de...","[A)II, B)I and II, C)I and III, D)II and III, ...",p@q = p^2 - pq=p(p-q).... so p@q will be zero ...,B,A. II\nB. I and II\nC. I and III\nD. II and II...,"For all numbers p and q, the operation @ is de...",I and II,p@q = p^2 - pq=p(p-q).... so p@q will be zero ...,Solve the following multiple-choice math reaso...
3,Carl is facing very difficult financial times ...,"[A)$1600, B)$2000, C)$2150, D)$2500, E)$12000]","Usually, you are given the annual rate of inte...",A,A. $1600\nB. $2000\nC. $2150\nD. $2500\nE. $12000,Carl is facing very difficult financial times ...,$1600,"Usually, you are given the annual rate of inte...",Solve the following multiple-choice math reaso...
4,The speed at which a man can row a boat in sti...,"[A)18 seconds, B)27 seconds, C)26 seconds, D)1...",Speed of the boat downstream = 25 +11\n= 36 km...,E,A. 18 seconds\nB. 27 seconds\nC. 26 seconds\nD...,The speed at which a man can row a boat in sti...,8 seconds,Speed of the boat downstream = 25 +11\n= 36 km...,Solve the following multiple-choice math reaso...


In [ ]:
print("Duplicate questions:",
      aqua_clean["question"].duplicated().sum())

Duplicate questions: 16543


In [ ]:
print(aqua_clean.loc[0, "question"])

print(aqua_clean.loc[0, "options"])

print(aqua_clean.loc[0, "correct"])

print(aqua_clean.loc[0, "rationale"])

Two friends plan to walk along a 43-km trail, starting at opposite ends of the trail at the same time. If Friend P's rate is 15% faster than Friend Q's, how many kilometers will Friend P have walked when they pass each other?
['A)21' 'B)21.5' 'C)22' 'D)22.5' 'E)23']
E
If Q complete x kilometers, then P completes 1.15x kilometers.
x + 1.15x = 43
2.15x=43
x = 43/2.15 = 20
Then P will have have walked 1.15*20=23 km.
The answer is E.


In [ ]:
def format_options(options):

    formatted = []

    for option in options:

        label = option[0]

        text = option[2:].strip()

        formatted.append(f"{label}. {text}")

    return "\n".join(formatted)

In [ ]:
aqua_clean["formatted_options"] = aqua_clean["options"].apply(format_options)

In [ ]:
print(aqua_clean.loc[0, "formatted_options"])

A. 21
B. 21.5
C. 22
D. 22.5
E. 23


In [ ]:
aqua_clean["input"] = (

    aqua_clean["question"]

    + "\n\nChoices:\n"

    + aqua_clean["formatted_options"]

)

dataset health


In [ ]:
print(aqua_clean.loc[0, "input"])

Two friends plan to walk along a 43-km trail, starting at opposite ends of the trail at the same time. If Friend P's rate is 15% faster than Friend Q's, how many kilometers will Friend P have walked when they pass each other?

Choices:
A. 21
B. 21.5
C. 22
D. 22.5
E. 23


In [ ]:
def extract_answer(row):

    answer_key = row["correct"]

    options = row["options"]

    for option in options:

        if option.startswith(answer_key):

            return option[2:].strip()

    return None

In [ ]:
aqua_clean["answer"] = aqua_clean.apply(
    extract_answer,
    axis=1
)

In [ ]:
print("Missing answers:")

print(aqua_clean["answer"].isnull().sum())

Missing answers:
0


In [ ]:
aqua_clean["reasoning"] = aqua_clean["rationale"]

In [ ]:
print(aqua_clean.loc[0, "reasoning"])



If Q complete x kilometers, then P completes 1.15x kilometers.
x + 1.15x = 43
2.15x=43
x = 43/2.15 = 20
Then P will have have walked 1.15*20=23 km.
The answer is E.


In [ ]:
aqua_clean["instruction"] = (
    "Solve the following multiple-choice math reasoning problem. "
    "Show your reasoning before giving the final answer."
)

In [ ]:
aqua_clean["dataset"] = "AQUA-RAT"

aqua_clean["task_type"] = "math_reasoning"

In [ ]:
aqua_clean = aqua_clean[
    [
        "instruction",
        "input",
        "reasoning",
        "answer",
        "dataset",
        "task_type"
    ]
]

In [ ]:
print(aqua_clean.shape)

aqua_clean.info()

print(aqua_clean.isnull().sum())

aqua_clean.head()

(97467, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97467 entries, 0 to 97466
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   question           97467 non-null  object
 1   options            97467 non-null  object
 2   rationale          97467 non-null  object
 3   correct            97467 non-null  object
 4   formatted_options  97467 non-null  object
 5   input              97467 non-null  object
 6   answer             97467 non-null  object
 7   reasoning          97467 non-null  object
 8   instruction        97467 non-null  object
dtypes: object(9)
memory usage: 6.7+ MB
question             0
options              0
rationale            0
correct              0
formatted_options    0
input                0
answer               0
reasoning            0
instruction          0
dtype: int64


,question,options,rationale,correct,formatted_options,input,answer,reasoning,instruction
0,"Two friends plan to walk along a 43-km trail, ...","[A)21, B)21.5, C)22, D)22.5, E)23]","If Q complete x kilometers, then P completes 1...",E,A. 21\nB. 21.5\nC. 22\nD. 22.5\nE. 23,"Two friends plan to walk along a 43-km trail, ...",23,"If Q complete x kilometers, then P completes 1...",Solve the following multiple-choice math reaso...
1,"In the coordinate plane, points (x, 1) and (5,...","[A)4 and 1, B)1 and 5, C)5 and 1, D)3 and 5, E...",Line k passes through the origin and has slope...,C,A. 4 and 1\nB. 1 and 5\nC. 5 and 1\nD. 3 and 5...,"In the coordinate plane, points (x, 1) and (5,...",5 and 1,Line k passes through the origin and has slope...,Solve the following multiple-choice math reaso...
2,"For all numbers p and q, the operation @ is de...","[A)II, B)I and II, C)I and III, D)II and III, ...",p@q = p^2 - pq=p(p-q).... so p@q will be zero ...,B,A. II\nB. I and II\nC. I and III\nD. II and II...,"For all numbers p and q, the operation @ is de...",I and II,p@q = p^2 - pq=p(p-q).... so p@q will be zero ...,Solve the following multiple-choice math reaso...
3,Carl is facing very difficult financial times ...,"[A)$1600, B)$2000, C)$2150, D)$2500, E)$12000]","Usually, you are given the annual rate of inte...",A,A. $1600\nB. $2000\nC. $2150\nD. $2500\nE. $12000,Carl is facing very difficult financial times ...,$1600,"Usually, you are given the annual rate of inte...",Solve the following multiple-choice math reaso...
4,The speed at which a man can row a boat in sti...,"[A)18 seconds, B)27 seconds, C)26 seconds, D)1...",Speed of the boat downstream = 25 +11\n= 36 km...,E,A. 18 seconds\nB. 27 seconds\nC. 26 seconds\nD...,The speed at which a man can row a boat in sti...,8 seconds,Speed of the boat downstream = 25 +11\n= 36 km...,Solve the following multiple-choice math reaso...


In [ ]:

aqua_clean["dataset"] = "AQUA-RAT"

aqua_clean["task_type"] = "math_reasoning"

print(aqua_clean.columns.tolist())

['question', 'options', 'rationale', 'correct', 'formatted_options', 'input', 'answer', 'reasoning', 'instruction', 'dataset', 'task_type']


In [ ]:
aqua_clean = aqua_clean[
    [
        "instruction",
        "input",
        "reasoning",
        "answer",
        "dataset",
        "task_type"
    ]
]

In [ ]:
print(aqua_clean.info())
print(aqua_clean.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97467 entries, 0 to 97466
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  97467 non-null  object
 1   input        97467 non-null  object
 2   reasoning    97467 non-null  object
 3   answer       97467 non-null  object
 4   dataset      97467 non-null  object
 5   task_type    97467 non-null  object
dtypes: object(6)
memory usage: 4.5+ MB
None
instruction    0
input          0
reasoning      0
answer         0
dataset        0
task_type      0
dtype: int64


In [ ]:
print("NaN answers:")
print(aqua_clean["answer"].isnull().sum())

print("\nEmpty string answers:")
print((aqua_clean["answer"] == "").sum())

print("\nWhitespace-only answers:")
print(
    aqua_clean["answer"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

NaN answers:
0

Empty string answers:
0

Whitespace-only answers:
0


In [ ]:
aqua_clean.to_csv(
    "aqua_rat_standardized.csv",
    index=False
)

In [ ]:
from google.colab import files

files.download("aqua_rat_standardized.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
check = pd.read_csv(
    "aqua_rat_standardized.csv",
    keep_default_na=False
)

print(check.isnull().sum())

instruction    0
input          0
reasoning      0
answer         0
dataset        0
task_type      0
dtype: int64


In [ ]:
print(check.info())
print(check.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97467 entries, 0 to 97466
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  97467 non-null  object
 1   input        97467 non-null  object
 2   reasoning    97467 non-null  object
 3   answer       97344 non-null  object
 4   dataset      97467 non-null  object
 5   task_type    97467 non-null  object
dtypes: object(6)
memory usage: 4.5+ MB
None
instruction      0
input            0
reasoning        0
answer         123
dataset          0
task_type        0
dtype: int64
